# Dimensionality Reduction

Examples of ways to systematically implement feature selection. Just remember there is no one "best" method - try what makes sense for your data and the question you are trying to answer.

## Feature Selection Methods

### Best Subset Selection

The Best Subset Selection method considers creating a model for every possible combination of predictors, then using some model measure like adjusted  to compare and select the best model. Here is the algorithm description from the textbook:

![Best Subset Selection](images/best_subset.png)

This method is problematic though - the number of models we need to create with this algorithm increases **even faster than exponential growth**. The notation $\binom{p}{k}$ is read as "$p$ choose $k$" and describes the number of models with exactly  predictors from the original . This means the total number of models with predictors we need to create is $\sum_{k=1}^{p} \binom{p}{k}$, and then we create one more to compare these with the null model $M_0$.

#### Example Framework to Run Best Subset Selection

To implement a best subset selection in Python, consider creating a function you can use in a loop. 

In [ ]:
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

# This function takes in a collection of predictors and creates a train-test split
def create_train_test_split(X_base, y_base):
    X_base_with_constant = sm.add_constant(X_base)
    X_train, X_test, y_train, y_test = train_test_split(X_base_with_constant, y_base, test_size=0.8)
    return [X_train, X_test, y_train, y_test]

# This function creates a logistic regression on the provided training data
def create_linear_regression(y_train, X_train):
    linear_model = sm.OLS(y_train, X_train)
    linear_regression = linear_model.fit()
    return linear_regression

Now you can create a loop that creates a logit regression for each subset. Here's an outline of what that could look like. 

In [ ]:
import pandas as pd

list_of_eval_measures = []

# Create the null model separately and add it to the 'list_of_eval_measures'

# You will need to define this list manually or utilize functions like itertools.combinations(iterable, r)
## https://docs.python.org/3/library/itertools.html#itertools.combinations
all_models_list = [model_1, model_2, model_3, ...] 

for X_set in all_models_list:
    X_train, X_test, y_train, y_test = create_train_test_split(X_set, y_set)
    linear_model = create_linear_regress(y_train, X_train)
    eval_measure = # You probably want to report a value here
    list_of_eval_measures.append(eval_measure)

# Creates a DataFrame to organize our results
eval_df = pd.DataFrame(list_of_eval_measures)
# Adds the names of the model variables to the first column of the DataFrame
eval_df.insert(0, 'model variables', ['null model']+all_models_list)

### Forward Stepwise Selection

Instead of going through every single possible model, we start with a model with no predictors, call it $M_0$. We then look at every possible model adding 1 predictor to this model and choose the best model, call it $M_1$. We do this same processing of taking $M_1$, adding each remaining predictor to it, and determining which of these are the best model, calling it $M_2$. We keep this going until we've gone through all possible predictors. 

![Forward Stepwise Selection](images/forward_step.png)

How does this compare to the brute force method? If we did the brute force method with 3 predictors (call them a, b, c, d, e), we would need to check:

- $M_0$ (no predictors)
- a, b, c, d, e,
- ab, ac, ad, ae, bc, bd, be, cd, ce, de, 
- abc, abd, abe, acd, ace, ade, bcd, bce, bde, cde, 
- abcd, abce, abde, acde, bcde
- abcde

*32 total models to build and compare*

With the forward stepwise algorithm, we would need to check:

- $M_0$ (no predictors)
- a, b, c, d, e [we have a best variable predictor, let's say c]
- ca, cb, cd, ce [we have a best model now, let's say cd]
- cda, cdb, cde [we have a best model now, let's say cda]
- cdab, cdae [we have a best model now, let's say it was cdae]

16 total models to build, 6 models to compare to determine the best

#### Example Framework to Run Forward Stepwise Selection

Now our loop needs to make an evaluation within the loop, but otherwise we can use a lot of what we did for the best subset selection method.

In [ ]:
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

# This function takes in a collection of predictors and creates a train-test split
def create_train_test_split(X_base, y_base):
    X_base_with_constant = sm.add_constant(X_base)
    X_train, X_test, y_train, y_test = train_test_split(X_base_with_constant, y_base, test_size=0.8)
    return [X_train, X_test, y_train, y_test]

# This function creates a logistic regression on the provided training data
def create_linear_regression(y_train, X_train):
    linear_model = sm.OLS(y_train, X_train)
    linear_regression = linear_model.fit()
    return linear_regression

In [ ]:
import pandas as pd

# This time we will create a list of dictionaries for the best model at each step
list_of_final_model_dicts = []

null_dict = {
    'model_name': 'null model',
    'model_variables': 'None',
    'model X_set': , # This should be a column of 1s the length of your full X_set
    'model': , # create null model and place it here
    'eval measure': # report evalulation measure of your choice here
}

list_of_final_model_dicts.append(null_dict)

all_vars_list = # create a list of all variables. If your data was loaded as a DataFrame, you could use df.keys()

current_model_dict = null_dict

# This time, we will start with the list and pop off variables as we include them in our model
while all_vars_list != []:
    temp_list_of_models_to_compare = [] # Resets a fresh list to compare during this loop
    
    for remaining_var in all_vars_list:
        current_X_set = current_model_dict['model X_set'] # Defines our base X_set that we will add to
        temp_X_set = # Think about how to combine the current X_set and the column of data associated with 'remaining_var'
        X_train, X_test, y_train, y_test = create_train_test_split(temp_X_set, y_set)

        temp_linear_model = create_linear_regression(y_train, X_train)
        temp_eval_measure = # You probably want to report a value here

        temp_dict = {
            'model_name': , # Think about a way to dynamically name your model using the length of 'all_vars_list'
            'model_variables': , # How can we add 'remaining_var' to the other variables in the 'current_model'?
            'model X_set': temp_X_set, 
            'model': temp_linear_model, 
            'eval measure': temp_eval_measure
        }
        temp_list_of_models_to_compare.append(temp_dict)

    # We've now created dicts of all the models we want to compare
    # Search through this list to find the highest eval measure model
    current_model_dict = # This is the dict in temp_list_of_models_to_compare with the highest eval measure
    list_of_final_model_dicts.append(current_model_dict)

    # We now need to remove the variable that has been selected to be added to create the current_model_dict
    newly_added_var = # How can we access the newly added variable so we can remove it from all_vars_list?
    all_vars_list = all_vars_list.remove(newly_added_var)

# Creates a DataFrame to organize our results
forward_stepwise_df = pd.DataFrame(list_of_final_model_dicts)

# Think about how to report the best model and if any columns need to be added or dropped

### Backward Stepwise Selection

Like the Forward Stepwise selection, we can instead work backwards from the model with all predictors included and compare what happens when we take away 1 predictor each time. 

![Backward Stepwise Selection](images/backward_stepwise.png)

The total number of models to create with the Forward/Backward Stepwise method is 

$$ \sum_{k=0}^{p-1} (p-k) = 1 + \frac{p(p+1)}{2} $$

where $p$ is the number of predictors and $k$ is the step.

#### Example Framework to Run Backward Stepwise Selection

The Backward Stepwise Selection will look almost identical to the Forward Stepwise Selection except for two key places: We need to begin with the full model (rather than the null model) and we need to remove a column of data to create temp_X_set rather than add a column of data. 

### T-Test Significance and the Bonferroni Correction

Since regression reports commonly provide a t-test on individual predictor variables, you could set a significance level cutoff and use all predictor variables that meet the cutoff threshold. But this introduces an issue that occurs when conducting many hypotheses tests at once! The textbook uses a coin flip to explain:

> Suppose that we flip 1,024 fair coins ten times each. Then we would expect (on average) one coin to come up all tails. But the chance of a single coin coming up all tails is $\frac{1}{1024} \approx 0.00098$ or $0.098\%$, so conducting a hypothesis test on this single coin being fair would lead to a p-value below 0.001 and we would thus reject the null hypothesis that this coin is fair.

In other words -- when testing a huge number of null hypotheses, we are bound to get some very small p-values by chance. 

There are multiple ways to correct for this, but we will use The Bonferroni Correction. 

#### The Bonferroni Correction

Given a significance level $\alpha$ and testing $m$ null hypotheses, the rejection threshold should be $\frac{\alpha}{m}$.

#### Using T-Test Significance and the Bonferroni Correction

In Python, you can access the p-values for each predictor with *model.pvalues*, coefficients with *model.params*.

In [ ]:
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

# This function takes in a collection of predictors and creates a train-test split
def create_train_test_split(X_base, y_base):
    X_base_with_constant = sm.add_constant(X_base)
    X_train, X_test, y_train, y_test = train_test_split(X_base_with_constant, y_base, test_size=0.8)
    return [X_train, X_test, y_train, y_test]

# This function creates a logistic regression on the provided training data
def create_linear_regression(y_train, X_train):
    linear_model = sm.OLS(y_train, X_train)
    linear_regression = linear_model.fit()
    return linear_regression

In [ ]:
# make an X_set and y_set of your data
X_set = 
y_set = 

# Split and run a regression on all variables
X_train, X_test, y_train, y_test = create_train_test_split(X_set, y_set)
full_linear_model = create_linear_regression(y_train, X_train)

# Collect all p-values to make comparisons
all_p_values = full_linear_model.pvalues
var_p_values = all_p_values.pop(0) # Think about why we would remove the first p-value...
# It may be more efficient to create a dictionary for each p-value here
# Include the predictor name and coefficient in the dictionary

# Create your correction threshold
alpha_sig_level = # define your alpha
bonferroni_correction_threshold = alpha_sig_level / len(var_p_values)

# Loop through your p-values to compare them to your correction threshold, keeping the ones that meet the threshold
sig_vars_list = []
for temp_p_value in var_p_values:
    if temp_p_value >= bonferroni_correction_threshold:
        temp_sig_var = # We need to add that predictor variable name to a list of vars we want to keep...
        sig_vars_list.append(temp_sig_var)

# Reduce your X_set to just the significant variables and run a final model
final_X_set = X_set[sig_vars_list]
X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = create_train_test_split(final_X_set, y_set)
final_linear_model = create_linear_regression(y_train_reduced, X_train_reduced)

### Ridge and Lasso

Ridge Regression is similar to the Ordinary Least Squares (OLS) Regression in that it wants to minimize the residual sum of squares (RSS). However, we also apply a penalty (known as a shrinkage penalty) to regression coefficients and minimize this combination. 

$$ \text{RSS}_{\text{ridge}} = RSS + \lambda \sum_{j=1}^p \beta_j^2 $$

This penalty is smallest when $\beta_1, \beta_2, \ldots, \beta_p$ are all close to zero, so this forces the model to prioritize shrinking the coefficients. This shrinks all coefficients towards 0, which effectively forces coefficients that were small in the OLS to be 0 and drop from the regression. We should use Ridge Regression when we believe most of the predictive variables are needed for an accurate model.

Lasso Regression is nearly identical to Ridge Regression with a different shrinkage penalty calculation:

$$ \text{RSS}_{\text{lasso}} = RSS + \lambda \sum_{j=1}^p | \, \beta_j \, | $$

This penalty is smallest when most of $\beta_1, \beta_2, \ldots, \beta_p$ are exactly zero, so this forces the model to prioritize removing all but the most important coefficients. This effectively drops most coefficients from the final model. We should use Lasso Regression when we believe there are just a few predictive variables needed for an accurate model.

#### Number of Models to Build

Changing $\lambda$ changes the Ridge/Lasso model! When $\lambda=0$ the shrinkage penalty becomes 0 and it is the OLS Regression. As $\lambda$ increases to $\infty$, all coefficients will eventually go to 0 and it becomes the null model  that only includes the intercept and no predictive variables. However, each choice of $\lambda$ gives one set of predictive coefficients $\beta_1, \beta_2, \ldots, \beta_p$, so the number of models we need to create to find the best model does not depend on the number of predictors we have! 

#### Examples of Ridge and Lasso Regressions

Implementing Ridge and Lasso regressions are frequently a matter of calling the specific function. Note that statsmodels.api has not implemented .summary() for ridge and lasso regressions. You would need to use .predict(exog=X_test) and manually calculate your evaluation measure. Instead, consider using Scikit-Learn's functions for Ridge and Lasso. 

In [50]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
import os


# Load Data
diabetes_file_path = os.path.join(os.getcwd(), 'datasets', 'Diabetes_and_LifeStyle_Dataset.csv')
diabetes_df = pd.read_csv(diabetes_file_path)

diabetes_response_var = diabetes_df['diagnosed_diabetes']
# For now, we will take numeric variables only
numeric_vars = ['Age', 'alcohol_consumption_per_week',
       'physical_activity_minutes_per_week', 'diet_score',
       'sleep_hours_per_day', 'screen_time_hours_per_day',
       'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp',
       'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol',
       'ldl_cholesterol', 'triglycerides', 'glucose_fasting',
       'glucose_postprandial', 'insulin_level', 'hba1c',
       'diabetes_risk_score']

diabetes_predictor_vars = diabetes_df[numeric_vars]

X_train, X_test, y_train, y_test = train_test_split(diabetes_predictor_vars, diabetes_response_var, test_size=0.8)

In [45]:
lin_reg = LinearRegression().fit(X_train, y_train)
ridge_reg = Lasso().fit(X_train, y_train)
lasso_reg = Ridge().fit(X_train, y_train)

print(f'Unstandardized Linear Regression R2: {round(lin_reg.score(X_test, y_test), 3)}')
print(f'L1-Standardized Linear Regression (Lasso) R2: {round(lasso_reg.score(X_test, y_test), 3)}')
print(f'L2-Standardized Linear Regression (Ridge) R2: {round(ridge_reg.score(X_test, y_test), 3)}')

Unstandardized Linear Regression R2: 0.465
L1-Standardized Linear Regression (Lasso) R2: 0.465
L2-Standardized Linear Regression (Ridge) R2: 0.403


## Feature Extraction Methods

### Principal Component Analysis

Principal Component Analysis is a technique for combining multiple predictive variables by creating "principal components" **without using the response variable**. In the figure from the book below, we are creating a new predictive variable on the axis "1st principal component" by finding the oblique line that maximizes variance, which ends up being the line of best fit. We then project all points onto that line as a way of estimating the two-dimensional points with a one-dimensional line. 

![PCA Book Example](images/pca_example.png)

We can create as many principal components as we have predictors.

#### Principal Component Regression

PCA has the surprising property that each subsequent principal component is "less important" than the next, so in practice you only need a few principal components to accurately create a regression model! And since each principal component is created using a combination of all predictors, we never exclude any predictors like the Feature Selection methods. The textbook provides an example of using PCR versus Ridge and Lasso regressions on a dataset of 50 observations and 45 predictors. 

![PCR vs Freature Selection Methods](images/pcr_vs_ridge_lasso.png)

#### Conducting a PCR

Scikit-Learn includes PCA: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

To incorporate PCA into a regression, you transform your predictor variables X using *PCA(n_components=n).fit_transform(X)* where $n$ is the number of principal components you want to use in your regression model. It is common to create a PCR for various $n$ and plot them like Figure 6.19 to determine which $n$ to use as your final model. 

In [ ]:
import pandas as pd
import os

# Load Data
diabetes_file_path = os.path.join(os.getcwd(), 'datasets', 'Diabetes_and_LifeStyle_Dataset.csv')
diabetes_df = pd.read_csv(diabetes_file_path)

diabetes_response_var = diabetes_df['diagnosed_diabetes']
# For now, we will take numeric variables only
numeric_vars = ['Age', 'alcohol_consumption_per_week',
       'physical_activity_minutes_per_week', 'diet_score',
       'sleep_hours_per_day', 'screen_time_hours_per_day',
       'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp',
       'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol',
       'ldl_cholesterol', 'triglycerides', 'glucose_fasting',
       'glucose_postprandial', 'insulin_level', 'hba1c',
       'diabetes_risk_score']

diabetes_predictor_vars = diabetes_df[numeric_vars]

In [97]:
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

pca_dim_redux = PCA(n_components=4) # Change n to see how it changes the model's score
transformed_X = pca_dim_redux.fit_transform(diabetes_predictor_vars) # Notice we don't use y_train since PCA does NOT consider the reponse variable when building the model
X_train, X_test, y_train, y_test = train_test_split(transformed_X, diabetes_response_var, test_size=0.8) # We've now used PCA to transform our X set and can proceed with a linear regression

pcr = LinearRegression().fit(X_train, y_train)
y_pred = pd.DataFrame(pcr.predict(X_test))

print(f'PCA-Transformed Linear Regression R2: {round(r2_score(y_test, y_pred), 3)}')

PCA-Transformed Linear Regression R2: 0.417


### Partial Least Squares

PLS also identifies linear combinations of features, but includes the response variable when attempting to determine the line of best fit. Figure 6.21 shows an example of both the first PLS line and PCA line.

![Partial Least Squares Graph Example](images/pls_graph.png)

The mathematics of conducting PLS is beyond the scope of this course but is included in subsection 6.3.2 for interested students. 

#### Difference between PCA and PLS

The PLS method is used to decrease bias in our model, though it does so at the cost of increasing variance. As a reminder:

- **Variance** refers to how much the model would change if we used a different training data set
- **Bias** refers to error introduced by approximating a real-life problem using a simpler model

PLS is common with specific types of data (such as digitized spectrometry signals in chemometrics), but often performs similarly to a ridge regression or PCR. 

Scikit-Learn has a section with an example describing the differences between PCR and PLS Regression: https://scikit-learn.org/stable/auto_examples/cross_decomposition/plot_pcr_vs_pls.html#sphx-glr-auto-examples-cross-decomposition-plot-pcr-vs-pls-py. The most important part of the example is the description near the end:

> On the other hand, the PLS regressor manages to capture the effect of the direction with the lowest variance, thanks to its use of target information during the transformation: it can recognize that this direction is actually the most predictive. We note that the first PLS component is negatively correlated with the target, which comes from the fact that the signs of eigenvectors are arbitrary.

What this is stating is that PLS can create the principal components based on the training data and simultaneously consider how predictive these principal components are for the model! 

#### Example with PLS

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Load Data
diabetes_file_path = os.path.join(os.getcwd(), 'datasets', 'Diabetes_and_LifeStyle_Dataset.csv')
diabetes_df = pd.read_csv(diabetes_file_path)

diabetes_response_var = diabetes_df['diagnosed_diabetes']
# For now, we will take numeric variables only
numeric_vars = ['Age', 'alcohol_consumption_per_week',
       'physical_activity_minutes_per_week', 'diet_score',
       'sleep_hours_per_day', 'screen_time_hours_per_day',
       'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history', 'bmi', 'waist_to_hip_ratio', 'systolic_bp',
       'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol',
       'ldl_cholesterol', 'triglycerides', 'glucose_fasting',
       'glucose_postprandial', 'insulin_level', 'hba1c',
       'diabetes_risk_score']

diabetes_predictor_vars = diabetes_df[numeric_vars]
X_train, X_test, y_train, y_test = train_test_split(diabetes_predictor_vars, diabetes_response_var, test_size=0.8) 


In [100]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score

pls_reg = PLSRegression(n_components=4).fit(X_train, y_train) # Change n to see how it changes the model's score
y_pred = pd.DataFrame(pls_reg.predict(X_test))

print(f'PLS-Transformed Linear Regression R2: {round(r2_score(y_test, y_pred), 3)}')

PLS-Transformed Linear Regression R2: 0.417


### Term Frequency-Inverse Document Frequency Vectorizer

While PCA and PLS are the most common feature extraction methods, numerous others exist to deal with specific types of data. With how popular Generative AI is, it would be good to look at one such feature extraction method specific to text: **Term Frequency - Inverse Document Frequency Vectorizer**. 

When talking about GenAI, you've likely heard the word "token". Tokens are the smallest unit of data to represent words, parts of words, or even single characters in text. We can extract meaning from these tokens in a sentence by creating a dictionary of meaning associated with the token. Scikit-learn provides an example: https://scikit-learn.org/stable/modules/feature_extraction.html

![POS Example](images/pos_example.png)

Term-Frequency-Inverse Document Frequency (Tf-idf) is a method to transform tokens based on how frequently they appear in a text. The "inverse" component tells us the less frequently the token appears, the more important it is. This lets us push down the importance of common tokens like "a", "and", "the" in our model. 

In the Scikit-Learn example of classifying text documents (https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html#sphx-glr-auto-examples-text-plot-document-classification-20newsgroups-py), we see Tf-idf used to extract features in two lines:

```
vectorizer = TfidfVectorizer(sublinear_tf=True, max_df=0.5, min_df=5, stop_words="english") 
X_train = vectorizer.fit_transform(data_train.data)
```